In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
import numpy as np
from scipy import stats as scipy_stats
import json

In [2]:
MARKETS_FILE = Path("../../markets.json")
BASE = Path("../../data")

with open(MARKETS_FILE) as f:
    config = json.load(f)

INITIAL_INVESTMENT = config["initial_investment"]
MONTHLY_INVESTMENT = config["monthly_investment"]
MARKETS            = config["markets"]


RISK_FREE_ANNUAL = 0.0351          # ~10yr Treasury yield used as Rf
RISK_FREE_MONTHLY = (1 + RISK_FREE_ANNUAL) ** (1/12) - 1

# Crisis periods shaded on all time-series charts
CRISES = [
    ("2008-09", "2009-06", "GFC"),
    ("2020-02", "2020-04", "COVID"),
    ("2022-01", "2022-12", "Rate Hikes"),
]

In [3]:
PALETTE = ["#028090", "#F6C90E", "#E53E3E", "#38A169", "#6B46C1", "#C05621"]
plt.rcParams.update({
    "figure.facecolor" : "#F4F7FB",
    "axes.facecolor"   : "white",
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "axes.grid"        : True,
    "grid.alpha"       : 0.4,
    "grid.color"       : "#CBD5E0",
    "font.family"      : "sans-serif",
})

RISK_FREE_ANNUAL, RISK_FREE_MONTHLY

(0.0351, 0.0028789730064431307)

In [ ]:
rets = pd.read_csv(returns_file, index_col=0, parse_dates=True)
act_price = pd.read_csv(price_file, index_col=0, parse_dates=True)

In [ ]:
assets = rets.columns.tolist()
number_of_assets = len(assets)


In [ ]:
growth = (1 + rets).cumprod()

# ── Cell 3: Individual Asset Growth of $1 ───────────────────────────────────
growth = (1 + rets).cumprod()

fig, ax = plt.subplots(figsize=(13, 6))

for col, color in zip(growth.columns, PALETTE):
    ax.plot(growth.index, growth[col], label=col, color=color, linewidth=1.8)
    # Annotate final value
    ax.annotate(
        f"{col}  ${growth[col].iloc[-1]:.2f}",
        xy=(growth.index[-1], growth[col].iloc[-1]),
        xytext=(8, 0), textcoords="offset points",
        fontsize=8, color=color, va="center"
    )

# Shade crisis periods
crises = [
    ("2008-09", "2009-06", "GFC"),
    ("2020-02", "2020-04", "COVID"),
    ("2022-01", "2022-12", "Rate Hikes"),
]
for start, end, label in crises:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end),
               alpha=0.10, color="red", zorder=0)
    ax.text(pd.Timestamp(start), ax.get_ylim()[1] * 0.97, label,
            fontsize=7.5, color="#E53E3E", ha="left")

ax.axhline(1, color="#8896A5", linewidth=0.8, linestyle="--")
ax.set_title("Growth of $1 Investment (2007–2025)", fontsize=14, fontweight="bold", pad=14)
ax.set_xlabel("Date")
ax.set_ylabel("Portfolio Value ($)")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 4: Actual ETF Prices ────────────────────────────────────────────────
# Note: raw prices are on different scales (VGT >> BND);
# use the $1 growth chart above for true relative performance comparison.

fig, ax = plt.subplots(figsize=(13, 6))

for col, color in zip(act_price.columns, PALETTE):
    ax.plot(act_price.index, act_price[col], label=col, color=color, linewidth=1.6)

ax.set_title("ETF Price History (2007–2025)", fontsize=14, fontweight="bold", pad=14)
ax.set_xlabel("Date")
ax.set_ylabel("Price ($)")
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}"))
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,6))

for col in growth.columns:
    plt.plot(growth.index, growth[col], label=col)

plt.title("Growth of $1 Investment (2007–2025)")
plt.xlabel("Date")
plt.ylabel("Portfolio Value")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# ── Cell 5: Equal-Weight Portfolio ───────────────────────────────────────────
# Serves as the null-hypothesis baseline for the 5-strategy comparison.

eq_weights  = pd.Series(1 / number_of_assets, index=assets)
port_ret    = (rets * eq_weights).sum(axis=1)   # proper weighted sum
port_growth = (1 + port_ret).cumprod()
port_pct    = (port_growth - 1) * 100

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(port_pct.index, port_pct, color="#028090", linewidth=2, label="Equal Weight Portfolio")
ax.fill_between(port_pct.index, port_pct, 0, alpha=0.08, color="#028090")
ax.axhline(0, color="#8896A5", linewidth=0.8, linestyle="--")

for start, end, label in crises:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.10, color="red", zorder=0)

ax.set_title("Cumulative Performance — Equal Weight Portfolio", fontsize=14, fontweight="bold", pad=14)
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative Return (%)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Total return  : {port_pct.iloc[-1]:.1f}%")
print(f"Final $1 value: ${port_growth.iloc[-1]:.2f}")

In [ ]:
# ── Cell 6: Allocation vs Volatility ─────────────────────────────────────────
# FIX: data is monthly → annualize with sqrt(12), not sqrt(252)

weights_pct = eq_weights * 100
vol_pct     = rets.std() * np.sqrt(12) * 100   # ← correct: monthly → annual

stats = pd.DataFrame({
    "Allocation %" : weights_pct,
    "Volatility %" : vol_pct
})

fig, ax = plt.subplots(figsize=(12, 6))
x     = np.arange(number_of_assets)
width = 0.35

bars1 = ax.bar(x - width/2, stats["Allocation %"], width,
               label="Equal Allocation", color="#028090", alpha=0.85)
bars2 = ax.bar(x + width/2, stats["Volatility %"],  width,
               label="Ann. Volatility (monthly√12)", color="#F6C90E", alpha=0.85)

# Value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(assets)
ax.set_ylabel("Percent (%)")
ax.set_title("Equal-Weight Allocation vs Annualized Volatility",
             fontsize=14, fontweight="bold", pad=14)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("\n── Risk Profile ──")
print(stats.round(2))

In [ ]:
# ── Cell 7: Sharpe Ratios (annualized, correct units) ────────────────────────
# FIX: annualize both return and vol; use monthly Rf to match monthly data

def sharpe(returns_series, rf_monthly=RISK_FREE_MONTHLY):
    """Annualized Sharpe ratio from monthly returns."""
    ann_excess = (returns_series.mean() - rf_monthly) * 12
    ann_vol    = returns_series.std() * np.sqrt(12)
    return ann_excess / ann_vol if ann_vol != 0 else np.nan

sharpe_ratios = {col: sharpe(rets[col]) for col in assets}
sharpe_ratios["Equal_Weight"] = sharpe(port_ret)

sr = pd.Series(sharpe_ratios)
colors = ["#028090" if v >= 0 else "#E53E3E" for v in sr]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(sr.index, sr.values, color=colors, alpha=0.85, width=0.55)

for bar, val in zip(bars, sr.values):
    offset = 0.02 if val >= 0 else -0.06
    ax.text(bar.get_x() + bar.get_width()/2, val + offset,
            f"{val:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.axhline(0, color="#8896A5", linewidth=0.8, linestyle="--")
ax.set_ylabel("Sharpe Ratio (annualized)")
ax.set_title("Annualized Sharpe Ratios — Individual Assets + Equal-Weight Portfolio",
             fontsize=13, fontweight="bold", pad=14)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

print("\n── Sharpe Ratios ──")
for k, v in sharpe_ratios.items():
    print(f"  {k:<25} {v:.4f}")

In [ ]:
print(rets.mean() * 12 * 100)   # annualized mean return %
print(rets.std() * np.sqrt(12) * 100)  # annualized vol %
print(f"Rf used: {RISK_FREE_ANNUAL*100:.2f}%")

In [ ]:
# ── Cell 8: Drawdown Paths ────────────────────────────────────────────────────

def drawdown_series(returns_series):
    cum         = (1 + returns_series).cumprod()
    rolling_max = cum.cummax()
    return (cum - rolling_max) / rolling_max * 100   # in %

# Individual asset drawdowns
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharey=True)
axes = axes.flatten()

for i, (col, color) in enumerate(zip(assets, PALETTE)):
    dd = drawdown_series(rets[col])
    ax = axes[i]
    ax.fill_between(dd.index, dd, 0, alpha=0.45, color=color)
    ax.plot(dd.index, dd, color=color, linewidth=1.2)
    ax.axhline(0, color="#8896A5", linewidth=0.6)
    for start, end, label in crises:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.12, color="red", zorder=0)
    ax.set_title(col, fontweight="bold", fontsize=12)
    ax.set_ylabel("Drawdown (%)" if i % 3 == 0 else "")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    # Annotate max drawdown
    max_dd = dd.min()
    ax.annotate(f"Max: {max_dd:.1f}%",
                xy=(dd.idxmin(), max_dd),
                xytext=(10, -18), textcoords="offset points",
                fontsize=8, color=color,
                arrowprops=dict(arrowstyle="->", color=color, lw=0.8))

fig.suptitle("Individual Asset Drawdown Paths (2007–2025)",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# Portfolio-level drawdown
fig, ax = plt.subplots(figsize=(13, 5))
port_dd = drawdown_series(port_ret)
ax.fill_between(port_dd.index, port_dd, 0, alpha=0.35, color="#028090")
ax.plot(port_dd.index, port_dd, color="#028090", linewidth=2, label="Equal Weight Portfolio")
ax.axhline(0, color="#8896A5", linewidth=0.8)
for start, end, label in crises:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.10, color="red", zorder=0)
    ax.text(pd.Timestamp(start), -1, label, fontsize=8, color="#E53E3E", ha="left")
ax.set_title("Equal-Weight Portfolio Drawdown Path",
             fontsize=14, fontweight="bold", pad=14)
ax.set_ylabel("Drawdown (%)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# Summary
print("── Max Drawdown by Asset ──")
for col in assets:
    dd = drawdown_series(rets[col])
    print(f"  {col:<6}  {dd.min():.2f}%   (trough: {dd.idxmin().date()})")
print(f"  {'EQ-W':<6}  {drawdown_series(port_ret).min():.2f}%   (trough: {drawdown_series(port_ret).idxmin().date()})")

In [ ]:
# ── Cell 9: Rolling Bond–Equity Correlation ──────────────────────────────────
# 24-month rolling window (2 years of monthly data)

WINDOW = 24

bond_proxies   = ["BND", "TLT", "TIP"]
equity_proxies = ["VGT", "VEU"]

# Only include assets that exist in the data
bond_cols   = [c for c in bond_proxies   if c in assets]
equity_cols = [c for c in equity_proxies if c in assets]

fig, ax = plt.subplots(figsize=(13, 5))

pair_colors = ["#028090", "#F6C90E", "#6B46C1", "#E53E3E", "#38A169", "#C05621"]
ci = 0
for bond in bond_cols:
    for equity in equity_cols:
        roll_corr = rets[bond].rolling(WINDOW).corr(rets[equity])
        ax.plot(roll_corr.index, roll_corr,
                label=f"{bond} / {equity}",
                linewidth=1.6, color=pair_colors[ci % len(pair_colors)])
        ci += 1

ax.axhline(0, color="#1A202C", linewidth=1.0, linestyle="--")
ax.axhline(-0.3, color="#8896A5", linewidth=0.6, linestyle=":")

for start, end, label in crises:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.10, color="red", zorder=0)
    ax.text(pd.Timestamp(start), ax.get_ylim()[0] + 0.03, label,
            fontsize=7.5, color="#E53E3E", ha="left")

ax.set_title(f"Rolling {WINDOW}-Month Bond–Equity Correlation\n"
             "(Risk Parity assumes this stays negative — 2022 broke that)",
             fontsize=13, fontweight="bold", pad=12)
ax.set_ylabel("Pearson Correlation")
ax.set_ylim(-1, 1)
ax.legend(fontsize=9, loc="lower left")
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 10: Return Distributions ────────────────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

dist_stats = []

for i, (col, color) in enumerate(zip(assets, PALETTE)):
    r  = rets[col].dropna()
    ax = axes[i]

    ax.hist(r * 100, bins=35, color=color, alpha=0.75, edgecolor="white", linewidth=0.4)
    ax.axvline(r.mean() * 100, color="#1A202C", linewidth=1.5, linestyle="--", label="Mean")
    ax.axvline(0, color="#8896A5", linewidth=0.8, linestyle=":")

    from scipy import stats as scipy_stats
    skew = scipy_stats.skew(r)
    kurt = scipy_stats.kurtosis(r)   # excess kurtosis (normal = 0)

    ax.set_title(col, fontweight="bold", fontsize=12)
    ax.set_xlabel("Monthly Return (%)")
    ax.set_ylabel("Frequency" if i % 3 == 0 else "")
    ax.text(0.97, 0.95, f"Skew: {skew:.2f}\nKurt: {kurt:.2f}",
            transform=ax.transAxes, ha="right", va="top",
            fontsize=8.5, color="#4A5568",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

    dist_stats.append({"Asset": col, "Mean (%)": round(r.mean()*100, 3),
                       "Std (%)": round(r.std()*100, 3),
                       "Skewness": round(skew, 3), "Excess Kurt.": round(kurt, 3)})

fig.suptitle("Monthly Return Distributions (2007–2025)",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

dist_df = pd.DataFrame(dist_stats).set_index("Asset")
print("── Distribution Statistics ──")
print(dist_df.to_string())
print("\nNote: |Skew| > 0.5 or |Kurt| > 1 signal non-normality")
print("      → FSO (Full Scale Optimization) is designed to exploit these.")